<a href="https://colab.research.google.com/github/DmitriyKolesnikM8O/MOEX-Scripts/blob/main/MOEX%26EMA50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===========================================
# Поиск акций MOEX возле EMA50 (ЧАСОВОЙ ТАЙМФРЕЙМ)
# ===========================================

import requests
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import time


DISTANCE = 1.0          # Максимальное расстояние до EMA50 в %
HOURS_BACK = 200        # Сколько часов назад скачать (200 часов = ~8 дней)
INTERVAL = 60           # 60 минут (часовой график)
DELAY = 0.1

print("📊 Загружаем список акций...")

url_securities = "https://iss.moex.com/iss/engines/stock/markets/shares/securities.json"

params_securities = {
    "iss.meta": "off",
    "securities.columns": "SECID,SHORTNAME,STATUS"
}

response = requests.get(url_securities, params=params_securities)
data = response.json()

stocks = pd.DataFrame(
    data["securities"]["data"],
    columns=data["securities"]["columns"]
)

stocks = stocks[stocks["STATUS"] == "A"]
stocks = stocks.reset_index(drop=True)

print(f"✅ Всего найдено активных акций: {len(stocks)}")


def get_recent_candles(ticker, hours_back=200, interval=60):
    """
    Получает ПОСЛЕДНИЕ свечи за указанное количество часов.
    """

    end_time = datetime.now()
    start_time = end_time - timedelta(hours=hours_back)


    start_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
    end_str = end_time.strftime("%Y-%m-%d %H:%M:%S")

    url = f"https://iss.moex.com/iss/engines/stock/markets/shares/securities/{ticker}/candles.json"

    params = {
        "from": start_str,
        "till": end_str,
        "interval": interval,
        "iss.meta": "off"
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if "candles" not in data or "data" not in data["candles"]:
            return None

        candles = data["candles"]["data"]
        if len(candles) < 50:
            return None

        columns = data["candles"]["columns"]
        df = pd.DataFrame(candles, columns=columns)


        df["close"] = pd.to_numeric(df["close"], errors='coerce')
        df = df.dropna(subset=["close"])

        if len(df) < 50:
            return None


        if "begin" in df.columns:
            df["begin"] = pd.to_datetime(df["begin"])
            df = df.sort_values("begin")

        return df

    except Exception as e:
        return None


results = []

print(f"\n🔍 Начинаем анализ акций (последние {HOURS_BACK} часов)...\n")


for index, row in tqdm(stocks.iterrows(), total=len(stocks), desc="Обработка акций"):

    ticker = row["SECID"]
    name = row["SHORTNAME"]

    try:

        df = get_recent_candles(ticker, HOURS_BACK, INTERVAL)

        if df is None or len(df) < 50:
            continue


        current_price = df["close"].iloc[-1]


        ema50 = df["close"].ewm(span=50, adjust=False).mean().iloc[-1]


        distance = abs(current_price - ema50) / ema50 * 100

        if distance <= DISTANCE:
            results.append({
                "Тикер": ticker,
                "Компания": name,
                "Цена (руб)": round(current_price, 2),
                "EMA50 (руб)": round(ema50, 2),
                "Расстояние %": round(distance, 2),
                "Выше EMA": "✅" if current_price > ema50 else "❌",
                "Свечей": len(df)
            })

        time.sleep(DELAY)

    except Exception as e:
        continue


print("\n" + "="*80)

if results:
    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values("Расстояние %")

    print(f"✅ НАЙДЕНО АКЦИЙ ВОЗЛЕ EMA50: {len(result_df)}")
    print("="*80)
    print("\n📊 Результаты:\n")

    print(result_df.to_string(index=False))

    filename = f"акции_возле_ema50_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    result_df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"\n💾 Сохранен в: {filename}")

else:
    print("❌ Акций возле EMA50 не найдено.")

print("="*80)


def check_ticker(ticker_symbol):
    df = get_recent_candles(ticker_symbol, HOURS_BACK, INTERVAL)

    if df is None or len(df) < 50:
        print(f"❌ Недостаточно данных для {ticker_symbol}")
        return

    current_price = df["close"].iloc[-1]
    ema50 = df["close"].ewm(span=50, adjust=False).mean().iloc[-1]

    print(f"\n🔍 {ticker_symbol}:")
    print(f"   Цена: {current_price:.2f} руб")
    print(f"   EMA50: {ema50:.2f} руб")
    print(f"   Отклонение: {abs(current_price - ema50) / ema50 * 100:.2f}%")
    print(f"   {'🔺 Выше' if current_price > ema50 else '🔻 Ниже'} EMA50")


# check_ticker("ABRD")

📊 Загружаем список акций...
✅ Всего найдено активных акций: 762

🔍 Начинаем анализ акций (последние 200 часов)...



Обработка акций: 100%|██████████| 762/762 [08:10<00:00,  1.55it/s]



✅ НАЙДЕНО АКЦИЙ ВОЗЛЕ EMA50: 200

📊 Результаты:

       Тикер   Компания  Цена (руб)  EMA50 (руб)  Расстояние % Выше EMA  Свечей
        INFL   INFL ETF      144.78       144.78          0.00        ❌      55
        CNYM   CNYM ETF       11.39        11.39          0.01        ❌      88
RU000A10EVE8   ПАРУС-МВ      898.00       898.07          0.01        ❌      59
RU000A1099U0 ЗПИФСовр 9        6.64         6.64          0.01        ✅      55
        CNYM   CNYM ETF       11.39        11.39          0.01        ❌      88
        VDOR   VDOR ETF     1037.50      1037.18          0.03        ✅      59
        LEAS   Европлан      669.90       670.17          0.04        ❌     105
        AKMB   ETF AKMB        1.98         1.99          0.04        ❌     108
        LEAS   Европлан      669.90       670.17          0.04        ❌     105
        AMRH   AMRH ETF      162.90       162.99          0.05        ❌      66
        LQDT   LQDT ETF        2.05         2.05          0.05        